In [1]:
%pip install plotly
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 1. Load & filter
# =========================================================
df = pd.read_csv("Fresh Entry.csv")

df = df[df["employmentTypes"].isin(["Full Time", "Permanent"])]
df = df[df["positionLevels"] == "Fresh/entry level"]
df = df[df["metadata_newPostingDate"] >= 45047]          # 2023-05-01
df = df.drop_duplicates(subset="metadata_jobPostId")

# =========================================================
# 2. Clean salary (min/max -> midpoint, drop placeholders)
# =========================================================
for c in ["salary_minimum", "salary_maximum"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    (df["salary_type"] == "Monthly")
    & df["salary_minimum"].notna()
    & df["salary_maximum"].notna()
].copy()

# swap if min > max
swap = df["salary_minimum"] > df["salary_maximum"]
df.loc[swap, ["salary_minimum", "salary_maximum"]] = (
    df.loc[swap, ["salary_maximum", "salary_minimum"]].values
)

df["salary_mid"] = (df["salary_minimum"] + df["salary_maximum"]) / 2

# drop obvious placeholder salaries (e.g. 1, 10, 12) for FT fresh-entry monthly
df = df[df["salary_mid"] >= 800]

# =========================================================
# 3. Applications & repost
# =========================================================
df["applications"]  = pd.to_numeric(df["metadata_totalNumberJobApplication"],
                                     errors="coerce").fillna(0)
df["repost_count"]  = pd.to_numeric(df["metadata_repostCount"],
                                     errors="coerce").fillna(0)

# =========================================================
# 4. Explode categories (fractional weight, avoid double-count)
# =========================================================
df["categories_list"] = df["categories"].apply(json.loads)
df["n_cat"]           = df["categories_list"].apply(len)

ex = df.explode("categories_list").copy()
ex["sector"] = ex["categories_list"].apply(lambda x: x["category"])
ex["weight"] = 1.0 / ex["n_cat"]

# winsorize salary per sector (1st–99th pct) to tame remaining outliers
def _wins(s):
    if len(s) < 5:
        return s
    return s.clip(s.quantile(0.01), s.quantile(0.99))

ex["salary_mid_w"] = ex.groupby("sector")["salary_mid"].transform(_wins)

# =========================================================
# 5. Sector-level aggregation
# =========================================================
agg = ex.groupby("sector").agg(
    postings  = ("weight",     "sum"),     # weighted count
    salary_y  = ("salary_mid_w","median"),
    apps_x    = ("applications", "median"),
    repost    = ("repost_count", "median"),
).reset_index()

# keep top-N sectors by posting volume for readability
TOP_N = 12
top = agg.sort_values("postings", ascending=False).head(TOP_N).copy()

# quadrant thresholds = medians of displayed sectors
x_mid = top["apps_x"].median()
y_mid = top["salary_y"].median()

# plot bounds
xmin, xmax = 0, max(top["apps_x"].max() * 1.15, 1)
ymin = top["salary_y"].min() * 0.85
ymax = top["salary_y"].max() * 1.15

# =========================================================
# 6. Build figure
# =========================================================
fig = go.Figure()

# --- quadrant shading (below points) ---
fig.add_shape(type="rect", x0=xmin, x1=x_mid, y0=y_mid, y1=ymax,
              fillcolor="rgba(244,63,94,0.10)",  line_width=0, layer="below")   # talent gap
fig.add_shape(type="rect", x0=x_mid, x1=xmax, y0=y_mid, y1=ymax,
              fillcolor="rgba(59,130,246,0.10)", line_width=0, layer="below")   # competitive
fig.add_shape(type="rect", x0=x_mid, x1=xmax, y0=ymin, y1=y_mid,
              fillcolor="rgba(245,158,11,0.10)", line_width=0, layer="below")   # high competition
fig.add_shape(type="rect", x0=xmin, x1=x_mid, y0=ymin, y1=y_mid,
              fillcolor="rgba(107,114,128,0.10)", line_width=0, layer="below")  # undesirable

# --- quadrant divider lines ---
fig.add_shape(type="line", x0=x_mid, x1=x_mid, y0=ymin, y1=ymax,
              line=dict(dash="dash", color="gray", width=1))
fig.add_shape(type="line", x0=xmin, x1=xmax, y0=y_mid, y1=y_mid,
              line=dict(dash="dash", color="gray", width=1))

# --- bubbles ---
size = top["postings"] / top["postings"].max() * 60 + 15

fig.add_trace(go.Scatter(
    x=top["apps_x"],
    y=top["salary_y"],
    mode="markers+text",
    text=top["sector"],
    textposition="top center",
    textfont=dict(size=10),
    marker=dict(
        size=size,
        color=top["repost"],
        colorscale="Reds",
        showscale=True,
        colorbar=dict(title="Median<br>repost count"),
        line=dict(color="white", width=1),
        opacity=0.85,
    ),
    customdata=np.stack([top["postings"], top["repost"]], axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Median applications: %{x:.0f}<br>"
        "Median salary mid: S$%{y:,.0f}/mo<br>"
        "Weighted postings: %{customdata[0]:.1f}<br>"
        "Median repost: %{customdata[1]:.0f}<extra></extra>"
    ),
    name="",
))

# --- quadrant labels ---
annotations = [
    dict(x=xmin, y=ymax, xanchor="left",  yanchor="top",
         text="🎯 Talent gap<br>high pay · low applications · reposts",
         showarrow=False, align="left", font=dict(size=11, color="#be123c")),
    dict(x=xmax, y=ymax, xanchor="right", yanchor="top",
         text="🔥 Competitive & attractive<br>high pay · high applications",
         showarrow=False, align="right", font=dict(size=11, color="#1d4ed8")),
    dict(x=xmax, y=ymin, xanchor="right", yanchor="bottom",
         text="📣 High competition<br>low pay · high applications (volume hiring)",
         showarrow=False, align="right", font=dict(size=11, color="#b45309")),
    dict(x=xmin, y=ymin, xanchor="left",  yanchor="bottom",
         text="⚠️ Undesirable / poorly advertised<br>low pay · low applications",
         showarrow=False, align="left", font=dict(size=11, color="#4b5563")),
]

fig.update_layout(
    title="TalentGap Finder — where do fresh-entry sectors sit?",
    xaxis_title="Median applications per posting",
    yaxis_title="Median fresh-entry salary midpoint (S$/month)",
    template="plotly_white",
    height=650, width=1050,
    annotations=annotations,
    margin=dict(l=80, r=40, t=80, b=60),
)

fig.show()
# fig.write_html("talent_gap_bubble.html")
# fig.write_image("talent_gap_bubble.png", scale=2)   # needs kaleido

  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 475.5 kB/s  0:00:20 eta 0:00:01
Using cached narwhals-2.28.0-py3-none-any.whl (490 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 1. Load & Clean Data
# =========================================================
df = pd.read_csv("Fresh Entry.csv")

df = df[df["employmentTypes"].isin(["Full Time", "Permanent"])]
df = df[df["positionLevels"] == "Fresh/entry level"]
df = df[df["metadata_newPostingDate"] >= 45047]          # 2023-05-01
df = df.drop_duplicates(subset="metadata_jobPostId")

for c in ["salary_minimum", "salary_maximum"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df[(df["salary_type"] == "Monthly") & df["salary_minimum"].notna() & df["salary_maximum"].notna()].copy()

swap = df["salary_minimum"] > df["salary_maximum"]
df.loc[swap, ["salary_minimum", "salary_maximum"]] = df.loc[swap, ["salary_maximum", "salary_minimum"]].values
df["salary_mid"] = (df["salary_minimum"] + df["salary_maximum"]) / 2
df = df[df["salary_mid"] >= 800] # Drop placeholders

df["applications"] = pd.to_numeric(df["metadata_totalNumberJobApplication"], errors="coerce").fillna(0)

# =========================================================
# 2. Explode Categories & Map to Macro-Sectors
# =========================================================
df["categories_list"] = df["categories"].apply(json.loads)
df["n_cat"] = df["categories_list"].apply(len)

ex = df.explode("categories_list").copy()
ex["sector_raw"] = ex["categories_list"].apply(lambda x: x["category"])
ex["weight"] = 1.0 / ex["n_cat"]

# Define the mapping to reduce 40+ categories into clean macro-sectors
def get_macro_sector(cat):
    cat = cat.lower()
    if 'information technology' in cat or 'it' in cat: return 'IT / Tech'
    if 'engineering' in cat or 'manufacturing' in cat or 'precision' in cat: return 'Engineering & Mfg'
    if 'banking' in cat or 'accounting' in cat or 'finance' in cat or 'auditing' in cat: return 'Finance & Accounting'
    if 'sales' in cat or 'retail' in cat or 'customer service' in cat or 'wholesale' in cat: return 'Sales & Customer Service'
    if 'admin' in cat or 'secretarial' in cat or 'human resources' in cat or 'legal' in cat: return 'Admin, HR & Legal'
    if 'healthcare' in cat or 'medical' in cat or 'therapy' in cat: return 'Healthcare'
    if 'education' in cat or 'training' in cat: return 'Education'
    if 'f&b' in cat or 'hospitality' in cat or 'travel' in cat: return 'F&B & Hospitality'
    if 'logistics' in cat or 'supply chain' in cat or 'purchasing' in cat: return 'Logistics & Supply Chain'
    if 'building' in cat or 'construction' in cat or 'architecture' in cat or 'property' in cat: return 'Construction & Property'
    if 'design' in cat or 'advertising' in cat or 'media' in cat or 'marketing' in cat: return 'Creative & Marketing'
    return 'Others'

ex["macro_sector"] = ex["sector_raw"].apply(get_macro_sector)

# Winsorize salary per macro-sector
def _wins(s):
    if len(s) < 5: return s
    return s.clip(s.quantile(0.01), s.quantile(0.99))
ex["salary_mid_w"] = ex.groupby("macro_sector")["salary_mid"].transform(_wins)

# =========================================================
# 3. Aggregate to Macro-Sector Level
# =========================================================
agg = ex.groupby("macro_sector").agg(
    postings = ("weight", "sum"),                  # Bubble Size
    total_apps = ("applications", "sum"),
    salary_y = ("salary_mid_w", "median")
).reset_index()

# Calculate average applications per posting
agg["apps_x"] = agg["total_apps"] / agg["postings"]

# Drop tiny sectors that clutter the chart (e.g., < 5 postings)
agg = agg[agg["postings"] >= 5].copy()

# Quadrant thresholds
x_mid = agg["apps_x"].median()
y_mid = agg["salary_y"].median()

# Identify "Talent Gap"
agg["is_gap"] = (agg["apps_x"] <= x_mid) & (agg["salary_y"] >= y_mid)
gap_df = agg[agg["is_gap"]]
other_df = agg[~agg["is_gap"]]

# =========================================================
# 4. Build the Chart
# =========================================================
fig = go.Figure()

# A. Shaded "Sweet spot" background
fig.add_shape(
    type="rect", x0=0, x1=x_mid, y0=y_mid, y1=agg["salary_y"].max() * 1.05,
    fillcolor="rgba(118, 199, 183, 0.15)", line_width=0, layer="below"
)

# B. Dashed Quadrant Lines
fig.add_vline(x=x_mid, line_dash="dash", line_color="#999", line_width=1)
fig.add_hline(y=y_mid, line_dash="dash", line_color="#999", line_width=1)

# C. Create Traces
def create_trace(data, color, name):
    # Bubble size scaled to Postings Count
    sizes = (data["postings"] / data["postings"].max() * 60) + 15
    
    return go.Scatter(
        x=data["apps_x"], y=data["salary_y"],
        mode="markers+text",
        name=name,
        text=data["macro_sector"],
        textposition="top center",
        textfont=dict(size=12, color="#333"),
        marker=dict(size=sizes, color=color, line=dict(color="white", width=1), opacity=0.9),
        customdata=np.stack([data["postings"], data["total_apps"]], axis=-1),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Avg Applicants per Job: %{x:.1f}<br>"
            "Median Pay: S$%{y:,.0f}<br>"
            "Total Postings: %{customdata[0]:.0f}<br>"
            "Total Applications: %{customdata[1]:,.0f}<extra></extra>"
        )
    )

fig.add_trace(create_trace(gap_df, "#F47C6C", "Talent-gap sector"))
fig.add_trace(create_trace(other_df, "#76C7B7", "Other sector"))

# D. Sweet Spot Annotation
fig.add_annotation(
    x=0.2, y=agg["salary_y"].max() * 1.02,
    text="Sweet spot: good pay, few applicants",
    showarrow=False, xanchor="left", yanchor="top",
    font=dict(size=13, color="#555")
)

# E. Layout
fig.update_layout(
    xaxis_title="Average applicants per job (fewer = easier to get hired)",
    yaxis_title="Typical monthly pay (S$)",
    template="plotly_white",
    height=650, width=1050,
    legend=dict(orientation="h", y=1.1, x=0, xanchor="left", title="Bubble size = Postings volume"),
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.update_xaxes(range=[0, max(agg["apps_x"]) * 1.15])
fig.update_yaxes(
    tickprefix="S$", 
    tickformat=",.0f", 
    range=[agg["salary_y"].min()*0.9, agg["salary_y"].max()*1.05]
)

fig.show()

In [4]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 1. Load & Clean Data
# =========================================================
df = pd.read_csv("Fresh Entry.csv")

df = df[df["employmentTypes"].isin(["Full Time", "Permanent"])]
df = df[df["positionLevels"] == "Fresh/entry level"]
df = df[df["metadata_newPostingDate"] >= 45047]          # 2023-05-01
df = df.drop_duplicates(subset="metadata_jobPostId")

for c in ["salary_minimum", "salary_maximum"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df[(df["salary_type"] == "Monthly") & df["salary_minimum"].notna() & df["salary_maximum"].notna()].copy()

swap = df["salary_minimum"] > df["salary_maximum"]
df.loc[swap, ["salary_minimum", "salary_maximum"]] = df.loc[swap, ["salary_maximum", "salary_minimum"]].values
df["salary_mid"] = (df["salary_minimum"] + df["salary_maximum"]) / 2
df = df[df["salary_mid"] >= 800] # Drop placeholders

df["applications"] = pd.to_numeric(df["metadata_totalNumberJobApplication"], errors="coerce").fillna(0)

# =========================================================
# 2. Explode Categories & Map to Macro-Sectors
# =========================================================
df["categories_list"] = df["categories"].apply(json.loads)
df["n_cat"] = df["categories_list"].apply(len)

ex = df.explode("categories_list").copy()
ex["sector_raw"] = ex["categories_list"].apply(lambda x: x["category"])
ex["weight"] = 1.0 / ex["n_cat"]

def get_macro_sector(cat):
    cat = cat.lower()
    if 'information technology' in cat or 'it' in cat: return 'IT / Tech'
    if 'engineering' in cat or 'manufacturing' in cat or 'precision' in cat: return 'Engineering & Mfg'
    if 'banking' in cat or 'accounting' in cat or 'finance' in cat or 'auditing' in cat: return 'Finance & Accounting'
    if 'sales' in cat or 'retail' in cat or 'customer service' in cat or 'wholesale' in cat: return 'Sales & Customer Service'
    if 'admin' in cat or 'secretarial' in cat or 'human resources' in cat or 'legal' in cat: return 'Admin, HR & Legal'
    if 'healthcare' in cat or 'medical' in cat or 'therapy' in cat: return 'Healthcare'
    if 'education' in cat or 'training' in cat: return 'Education'
    if 'f&b' in cat or 'hospitality' in cat or 'travel' in cat: return 'F&B & Hospitality'
    if 'logistics' in cat or 'supply chain' in cat or 'purchasing' in cat: return 'Logistics & Supply Chain'
    if 'building' in cat or 'construction' in cat or 'architecture' in cat or 'property' in cat: return 'Construction & Property'
    if 'design' in cat or 'advertising' in cat or 'media' in cat or 'marketing' in cat: return 'Creative & Marketing'
    return 'Others'

ex["macro_sector"] = ex["sector_raw"].apply(get_macro_sector)

def _wins(s):
    if len(s) < 5: return s
    return s.clip(s.quantile(0.01), s.quantile(0.99))
ex["salary_mid_w"] = ex.groupby("macro_sector")["salary_mid"].transform(_wins)

# =========================================================
# 3. Aggregate to Macro-Sector Level
# =========================================================
agg = ex.groupby("macro_sector").agg(
    postings = ("weight", "sum"),                  # Bubble Size
    total_apps = ("applications", "sum"),
    salary_x = ("salary_mid_w", "median")          # Swapped to X
).reset_index()

# Calculate average applications per posting (Now Y)
agg["apps_y"] = agg["total_apps"] / agg["postings"]

# Drop tiny sectors
agg = agg[agg["postings"] >= 5].copy()

# Quadrant thresholds
x_mid = agg["salary_x"].median()  # Median Salary
y_mid = agg["apps_y"].median()    # Median Applicants

# Identify "Talent Gap" (High Salary, Low Applicants)
agg["is_gap"] = (agg["salary_x"] >= x_mid) & (agg["apps_y"] <= y_mid)
gap_df = agg[agg["is_gap"]]
other_df = agg[~agg["is_gap"]]

# =========================================================
# 4. Build the Chart (Swapped Axes)
# =========================================================
fig = go.Figure()

# A. Shaded "Sweet spot" background (Bottom Right: High X, Low Y)
fig.add_shape(
    type="rect", 
    x0=x_mid, x1=agg["salary_x"].max() * 1.05, 
    y0=0, y1=y_mid,
    fillcolor="rgba(118, 199, 183, 0.15)", line_width=0, layer="below"
)

# B. Dashed Quadrant Lines
fig.add_vline(x=x_mid, line_dash="dash", line_color="#999", line_width=1)
fig.add_hline(y=y_mid, line_dash="dash", line_color="#999", line_width=1)

# C. Create Traces
def create_trace(data, color, name):
    sizes = (data["postings"] / data["postings"].max() * 60) + 15
    
    return go.Scatter(
        x=data["salary_x"], y=data["apps_y"],   # X=Salary, Y=Applicants
        mode="markers+text",
        name=name,
        text=data["macro_sector"],
        textposition="top center",
        textfont=dict(size=12, color="#333"),
        marker=dict(size=sizes, color=color, line=dict(color="white", width=1), opacity=0.9),
        customdata=np.stack([data["postings"], data["total_apps"]], axis=-1),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Median Pay: S$%{x:,.0f}<br>"
            "Avg Applicants per Job: %{y:.1f}<br>"
            "Total Postings: %{customdata[0]:.0f}<br>"
            "Total Applications: %{customdata[1]:,.0f}<extra></extra>"
        )
    )

fig.add_trace(create_trace(gap_df, "#F47C6C", "Talent-gap sector"))
fig.add_trace(create_trace(other_df, "#76C7B7", "Other sector"))

# D. Sweet Spot Annotation (Positioned in Bottom Right)
fig.add_annotation(
    x=agg["salary_x"].max() * 1.02, y=0.1,
    text="Sweet spot: good pay, few applicants",
    showarrow=False, xanchor="right", yanchor="bottom",
    font=dict(size=13, color="#555")
)

# E. Layout & Axis Titles
fig.update_layout(
    xaxis_title="Typical monthly pay (S$)",
    yaxis_title="Average applicants per job (lower = easier to get hired)",
    template="plotly_white",
    height=650, width=1050,
    legend=dict(orientation="h", y=1.1, x=0, xanchor="left", title="Bubble size = Postings volume"),
    margin=dict(l=60, r=40, t=80, b=60)
)

# Format Axes
fig.update_xaxes(
    tickprefix="S$", 
    tickformat=",.0f", 
    range=[agg["salary_x"].min()*0.9, agg["salary_x"].max()*1.1]
)
fig.update_yaxes(
    range=[0, max(agg["apps_y"]) * 1.15]
)

fig.show()